# ROGII: Boundary Geometry and Score Concentration

The official pooled row-wise RMSE does not give every well equal influence. A well with a long missing suffix contributes more rows, and a long high-error suffix can dominate the score through both row count and squared error.

This notebook maps the exact supplied prediction boundary for all 773 training wells. It measures visible-prefix length, suffix length, MD and Z spans, anchor bias, per-well RMSE, row share, and SSE share. The goal is to design validation strata that resemble deployment and to understand where score movement actually comes from.

All target use is evaluation-only after the supplied boundary. The notebook builds no model beyond the last-visible-TVT anchor, creates no submission, and uses no leaderboard feedback.

In [ ]:
from pathlib import Path
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

DATA_CANDIDATES = [
    Path('/kaggle/input/competitions/rogii-wellbore-geology-prediction'),
    Path('/kaggle/input/rogii-wellbore-geology-prediction'),
    Path('data/raw/competition'),
    Path('../../data/raw/competition'),
]
DATA_ROOT = next((path for path in DATA_CANDIDATES if path.exists()), None)
if DATA_ROOT is None:
    raise FileNotFoundError('Competition data was not found')
TRAIN_DIR = DATA_ROOT / 'train'
paths = sorted(TRAIN_DIR.glob('*__horizontal_well.csv'))
print('Data root:', DATA_ROOT)
print('Training wells:', len(paths))

## Measure the deployment-shaped boundary

`TVT_input` must be finite on one prefix and missing on one contiguous suffix. The final visible value is repeated as the anchor prediction. For each well we retain row-level SSE before taking any square root; averaging per-well RMSE would estimate a different objective.

In [ ]:
def split_boundary(values):
    observed = values.notna().to_numpy()
    missing = np.flatnonzero(~observed)
    if len(missing) == 0 or missing[0] < 1 or observed[missing[0]:].any():
        raise ValueError('Expected one visible prefix and one contiguous missing suffix')
    return int(missing[0])

records = []
for path in paths:
    frame = pd.read_csv(path, usecols=['MD', 'Z', 'TVT', 'TVT_input'])
    boundary = split_boundary(frame['TVT_input'])
    md = frame['MD'].to_numpy(dtype=float)
    z = frame['Z'].to_numpy(dtype=float)
    truth = frame['TVT'].to_numpy(dtype=float)
    anchor = float(frame.loc[boundary - 1, 'TVT_input'])
    error = anchor - truth[boundary:]
    n_suffix = len(error)
    sse = float(error @ error)
    records.append({
        'well_id': path.name.removesuffix('__horizontal_well.csv'),
        'total_rows': len(frame),
        'prefix_rows': boundary,
        'suffix_rows': n_suffix,
        'suffix_fraction': n_suffix / len(frame),
        'prefix_md_span': float(md[boundary - 1] - md[0]),
        'suffix_md_span': float(md[-1] - md[boundary]),
        'prefix_z_span': float(z[:boundary].max() - z[:boundary].min()),
        'suffix_z_span': float(z[boundary:].max() - z[boundary:].min()),
        'anchor': anchor,
        'sse': sse,
        'rmse': math.sqrt(sse / n_suffix),
        'bias': float(error.mean()),
        'end_error': float(error[-1]),
        'truth_range': float(truth[boundary:].max() - truth[boundary:].min()),
    })

wells = pd.DataFrame(records)
total_rows = int(wells['suffix_rows'].sum())
total_sse = float(wells['sse'].sum())
wells['row_share'] = wells['suffix_rows'] / total_rows
wells['sse_share'] = wells['sse'] / total_sse
print('Evaluation rows:', total_rows)
print('Anchor pooled RMSE:', math.sqrt(total_sse / total_rows))
display(wells.head())

In [ ]:
geometry_columns = [
    'prefix_rows', 'suffix_rows', 'suffix_fraction',
    'prefix_md_span', 'suffix_md_span', 'prefix_z_span', 'suffix_z_span',
    'rmse', 'bias', 'truth_range',
]
display(wells[geometry_columns].describe(percentiles=[0.10, 0.25, 0.50, 0.75, 0.90, 0.95]).T)

## Suffix-length strata

Quantile bins contain roughly the same number of wells, not the same number of rows. The table therefore makes official-metric weighting visible: compare each bin's `well_share`, `row_share`, and `sse_share`.

In [ ]:
wells['suffix_decile'] = pd.qcut(wells['suffix_rows'], 10, duplicates='drop')
strata_rows = []
for interval, group in wells.groupby('suffix_decile', observed=True):
    strata_rows.append({
        'suffix_decile': str(interval),
        'wells': len(group),
        'well_share': len(group) / len(wells),
        'suffix_rows_min': int(group['suffix_rows'].min()),
        'suffix_rows_max': int(group['suffix_rows'].max()),
        'row_share': group['suffix_rows'].sum() / total_rows,
        'sse_share': group['sse'].sum() / total_sse,
        'pooled_rmse': math.sqrt(group['sse'].sum() / group['suffix_rows'].sum()),
        'macro_rmse': group['rmse'].mean(),
        'median_abs_bias': group['bias'].abs().median(),
    })
strata = pd.DataFrame(strata_rows)
display(strata.style.format({
    'well_share': '{:.1%}', 'row_share': '{:.1%}', 'sse_share': '{:.1%}',
    'pooled_rmse': '{:.2f}', 'macro_rmse': '{:.2f}', 'median_abs_bias': '{:.2f}',
}))

In [ ]:
longest = wells.nlargest(math.ceil(0.10 * len(wells)), 'suffix_rows')
highest_sse = wells.nlargest(math.ceil(0.10 * len(wells)), 'sse')
concentration = pd.Series({
    'longest 10% wells: row share': longest['row_share'].sum(),
    'longest 10% wells: SSE share': longest['sse_share'].sum(),
    'highest-SSE 10% wells: row share': highest_sse['row_share'].sum(),
    'highest-SSE 10% wells: SSE share': highest_sse['sse_share'].sum(),
    'macro well RMSE': wells['rmse'].mean(),
    'official pooled RMSE': math.sqrt(total_sse / total_rows),
})
display(concentration.to_frame('value').style.format('{:.4f}'))

## Geometry is a validation stratum, not a complete error model

Known-at-inference quantities such as suffix length and trajectory span are valid stratification features. They can reveal coverage gaps and runtime risk. They should not be mistaken for sufficient confidence gates: large branch errors can occur in ordinary-looking geometry.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
axes[0, 0].scatter(wells['suffix_rows'], wells['rmse'], s=12, alpha=0.45)
axes[0, 0].set(title='Anchor error versus suffix length', xlabel='suffix rows', ylabel='well RMSE')
axes[0, 1].scatter(wells['suffix_z_span'], wells['rmse'], s=12, alpha=0.45, color='#347a4a')
axes[0, 1].set(title='Anchor error versus suffix Z span', xlabel='suffix Z span', ylabel='well RMSE')
axes[1, 0].scatter(wells['suffix_rows'], wells['sse_share'], s=12, alpha=0.45, color='#c45a32')
axes[1, 0].set(title='Official score contribution', xlabel='suffix rows', ylabel='fraction of total SSE')
axes[1, 1].scatter(wells['bias'], wells['end_error'], s=12, alpha=0.45, color='#6b4c9a')
limit = max(wells['bias'].abs().max(), wells['end_error'].abs().max())
axes[1, 1].plot([-limit, limit], [-limit, limit], 'k--', linewidth=1)
axes[1, 1].set(title='Persistent level error', xlabel='mean suffix error', ylabel='final-row error')
plt.tight_layout()
plt.show()

correlations = wells[[
    'suffix_rows', 'suffix_fraction', 'suffix_md_span', 'suffix_z_span',
    'truth_range', 'rmse', 'sse_share',
]].corr(method='spearman')
display(correlations[['rmse', 'sse_share']].style.format('{:.3f}'))

In [ ]:
display(wells.nlargest(15, 'sse_share')[[
    'well_id', 'prefix_rows', 'suffix_rows', 'suffix_fraction',
    'suffix_md_span', 'suffix_z_span', 'rmse', 'bias', 'end_error',
    'row_share', 'sse_share',
]].style.format({
    'suffix_fraction': '{:.1%}', 'row_share': '{:.2%}', 'sse_share': '{:.2%}',
    'suffix_md_span': '{:.1f}', 'suffix_z_span': '{:.1f}',
    'rmse': '{:.2f}', 'bias': '{:.2f}', 'end_error': '{:.2f}',
}))

## Reusable validation checklist

1. Reproduce the exact supplied boundary; do not replace it with arbitrary random rows.
2. Accumulate SSE and row counts before taking pooled RMSE.
3. Report macro, p95, and worst-well RMSE beside the official metric.
4. Stratify folds and diagnostics by known boundary geometry: suffix rows, suffix fraction, MD span, and Z span.
5. Track row share and SSE share separately. Long wells have more metric weight; hard wells have more squared-error weight.
6. Audit the largest contributors individually. A global average rarely explains a wrong geological branch.

The boundary is part of the data-generating process, not just a place to split a CSV.